# A6: Naive RAG vs Contextual Retrieval



This notebook builds a domain-specific QA system grounded in the Transformers chapter and compares two retrieval strategies:

| Strategy | Description |
|---|---|
| Naive RAG | Standard chunking, embedding, and cosine similarity retrieval |
| Contextual Retrieval | LLM-prepended context per chunk before embedding and retrieval |

The full pipeline is: `transformer_jan26.pdf` -> text chunks -> embeddings -> retrieval -> LLM generation -> ROUGE evaluation.

## Setup and Imports

In [ ]:
import os
import re
import json
import math
import asyncio
import numpy as np
import torch
import fitz  # PyMuPDF
from transformers import AutoTokenizer, AutoModel
from rouge_score import rouge_scorer
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Groq client setup
from groq import Groq

# API key - set the GROQ_API_KEY environment variable or paste directly here
GROQ_API_KEY    = os.environ.get('GROQ_API_KEY', '')
groq_client     = Groq(api_key=GROQ_API_KEY)
GROQ_MODEL      = 'llama-3.3-70b-versatile'  # used for answer generation
GROQ_FAST_MODEL = 'llama-3.1-8b-instant'     # used for bulk chunk enrichment

print('All imports loaded successfully.')
print(f'Torch version  : {torch.__version__}')
print(f'PyMuPDF version: {fitz.__version__}')
print(f'Groq model     : {GROQ_MODEL}')
print(f'Groq fast model: {GROQ_FAST_MODEL}')

All imports loaded successfully.
Torch version  : 2.3.1
PyMuPDF version: 1.27.2.2
Groq model     : llama-3.3-70b-versatile
Groq fast model: llama-3.1-8b-instant


In [26]:
# Fix seed for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Use GPU if available, otherwise fall back to CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


---
# Task 1: Source Discovery and Data Preparation

## 1.1 Load and Extract Text from PDF

In [27]:
PDF_PATH = 'transformer_jan26.pdf'

def extract_text_from_pdf(pdf_path: str) -> str:
    """Extract and concatenate raw text from all pages of a PDF file."""
    doc = fitz.open(pdf_path)
    pages_text = []
    for page_num, page in enumerate(doc):
        text = page.get_text('text')
        pages_text.append(text)
    doc.close()
    return '\n'.join(pages_text)

def clean_text(text: str) -> str:
    """Apply basic cleaning to the extracted PDF text."""
    # Rejoin words broken by hyphenated line breaks
    text = re.sub(r'-(\n)\s*', '', text)
    # Collapse sequences of more than two blank lines
    text = re.sub(r'\n{3,}', '\n\n', text)
    # Remove stand-alone page numbers on their own line
    text = re.sub(r'\n\s*\d+\s*\n', '\n', text)
    # Collapse multiple spaces and tabs into a single space
    text = re.sub(r'[ \t]+', ' ', text)
    # Strip leading and trailing whitespace from each line
    lines = [line.strip() for line in text.split('\n')]
    text = '\n'.join(lines)
    return text.strip()

raw_text = extract_text_from_pdf(PDF_PATH)
clean = clean_text(raw_text)

print(f'Extracted {len(raw_text):,} characters from the PDF')
print(f'Cleaned text length: {len(clean):,} characters')
print('\nFirst 500 characters:')
print(clean[:500])

Extracted 17,908 characters from the PDF
Cleaned text length: 17,633 characters

First 500 characters:
Transformers
Introduction to Transformers

LLMs are built out of transformers
Transformer: a specific kind of network architecture, like a
fancier feedforward network, but based on attention
Attention Is All You Need
Ashish Vaswani⇤
Google Brain
avaswani@google.com
Noam Shazeer⇤
Google Brain
noam@google.com
Niki Parmar⇤
Google Research
nikip@google.com
Jakob Uszkoreit⇤
Google Research
usz@google.com
Llion Jones⇤
Google Research
llion@google.com
Aidan N. Gomez⇤†
University of Toronto
aidan@cs.tor


## 1.2 Chunking

The text is split into overlapping word-based chunks. A sliding window of 500 words with a 50-word overlap is used to preserve context across chunk boundaries.

In [28]:
def chunk_text(text: str, chunk_size: int = 500, overlap: int = 50) -> list[str]:
    """
    Split text into overlapping word-based chunks.
    chunk_size : number of words per chunk
    overlap    : number of words shared between consecutive chunks
    """
    words = text.split()
    chunks = []
    start = 0
    step  = chunk_size - overlap
    while start < len(words):
        end   = min(start + chunk_size, len(words))
        chunk = ' '.join(words[start:end])
        # Discard very short chunks that are unlikely to carry meaningful content
        if len(chunk.split()) > 50:
            chunks.append(chunk)
        start += step
    return chunks

chunks = chunk_text(clean, chunk_size=500, overlap=50)
print(f'Total chunks created: {len(chunks)}')
print(f'Average words per chunk: {np.mean([len(c.split()) for c in chunks]):.0f}')
print('\nChunk 0 preview (first 400 chars):')
print(chunks[0][:400])

Total chunks created: 8
Average words per chunk: 459

Chunk 0 preview (first 400 chars):
Transformers Introduction to Transformers LLMs are built out of transformers Transformer: a specific kind of network architecture, like a fancier feedforward network, but based on attention Attention Is All You Need Ashish Vaswani⇤ Google Brain avaswani@google.com Noam Shazeer⇤ Google Brain noam@google.com Niki Parmar⇤ Google Research nikip@google.com Jakob Uszkoreit⇤ Google Research usz@google.co


## 1.3 QA Pair Dataset

Twenty question-answer pairs covering the key concepts in Chapter 8 of `transformer_jan26.pdf`. These are used both as the retrieval queries and as reference answers for ROUGE evaluation.

In [29]:
QA_PAIRS = [
    {
        "question": "What problem does the Transformer architecture solve that RNNs struggled with?",
        "ground_truth_answer": "The Transformer architecture solves the sequential processing bottleneck of RNNs. RNNs process tokens one at a time, making parallelization difficult and creating challenges for learning long-range dependencies due to vanishing gradients. Transformers use self-attention to process all tokens in parallel and capture dependencies regardless of distance."
    },
    {
        "question": "What is self-attention and how does it work?",
        "ground_truth_answer": "Self-attention allows each token in a sequence to attend to all other tokens in the same sequence. For each token, it computes Query (Q), Key (K), and Value (V) vectors. The attention score between tokens is computed as the dot product of Q and K, scaled by the square root of the dimension, then passed through softmax. The output is a weighted sum of the V vectors, where higher-attended tokens contribute more."
    },
    {
        "question": "Why is the attention score scaled by the square root of the key dimension?",
        "ground_truth_answer": "The dot products of Q and K can grow large in magnitude when the key dimension dk is large. Large dot products push the softmax into regions with extremely small gradients, making training difficult. Scaling by 1/sqrt(dk) keeps the dot products in a range where the softmax function produces more useful gradients."
    },
    {
        "question": "What is multi-head attention and why is it used?",
        "ground_truth_answer": "Multi-head attention runs multiple self-attention operations (heads) in parallel with different learned linear projections of Q, K, and V. Each head can attend to different parts of the sequence and capture different types of relationships. The outputs of all heads are concatenated and projected, allowing the model to jointly attend to information from different representation subspaces."
    },
    {
        "question": "How does the Transformer handle word order since it uses no recurrence?",
        "ground_truth_answer": "The Transformer adds positional encodings to the input embeddings to inject information about the position of each token. The original paper uses sinusoidal functions of different frequencies for each position and dimension. These encodings allow the model to distinguish tokens by their position, since self-attention itself is permutation-invariant."
    },
    {
        "question": "Describe the encoder-decoder structure of the original Transformer.",
        "ground_truth_answer": "The Transformer encoder is a stack of N identical layers, each with two sub-layers: multi-head self-attention and a position-wise feed-forward network, both followed by residual connections and layer normalization. The decoder is also a stack of N layers with three sub-layers: masked multi-head self-attention (to prevent attending to future tokens), encoder-decoder cross-attention, and a feed-forward network, again with residual connections and layer normalization."
    },
    {
        "question": "What is the purpose of masked self-attention in the Transformer decoder?",
        "ground_truth_answer": "Masked self-attention in the decoder prevents positions from attending to subsequent (future) positions. During training, the entire target sequence is fed in at once, but masking ensures that the prediction for position i can only depend on tokens at positions less than i. This maintains the autoregressive property required for sequence generation."
    },
    {
        "question": "What is the role of the feed-forward sub-layer in each Transformer layer?",
        "ground_truth_answer": "The position-wise feed-forward network (FFN) in each Transformer layer applies the same two linear transformations with a ReLU activation to each position independently: FFN(x) = max(0, xW1+b1)W2+b2. It adds non-linearity and increases the model's capacity to learn complex mappings after the attention operation has gathered information across positions."
    },
    {
        "question": "What are residual connections and layer normalization used for in the Transformer?",
        "ground_truth_answer": "Residual connections add the input of a sub-layer directly to its output (x + Sublayer(x)), allowing gradients to flow more easily during backpropagation and helping train deep networks. Layer normalization is then applied to stabilize activations by normalizing across the feature dimension, improving training speed and stability."
    },
    {
        "question": "What is cross-attention in the Transformer decoder and what does it attend to?",
        "ground_truth_answer": "Cross-attention (encoder-decoder attention) is the second sub-layer in the Transformer decoder. The Queries come from the decoder's previous sub-layer, while the Keys and Values come from the encoder's output. This allows each decoder position to attend over all positions in the input sequence, enabling the decoder to incorporate relevant source information when generating each output token."
    },
    {
        "question": "How are attention weights computed mathematically in scaled dot-product attention?",
        "ground_truth_answer": "Scaled dot-product attention is computed as: Attention(Q,K,V) = softmax(QK^T / sqrt(dk)) * V, where Q is the query matrix, K is the key matrix, V is the value matrix, and dk is the dimension of the keys. The dot products are scaled by 1/sqrt(dk) before applying softmax to obtain normalized attention weights, which are then used to compute a weighted sum of the values."
    },
    {
        "question": "What are the three types of attention used in the Transformer?",
        "ground_truth_answer": "The Transformer uses three types of attention: (1) Encoder self-attention, where each position in the encoder attends to all positions in the encoder input; (2) Decoder masked self-attention, where each position in the decoder attends to all positions up to and including that position; and (3) Encoder-decoder cross-attention, where decoder positions attend to all positions in the encoder output."
    },
    {
        "question": "What are sinusoidal positional encodings and how are they formulated?",
        "ground_truth_answer": "Sinusoidal positional encodings assign each position a unique vector using sine and cosine functions of different frequencies: PE(pos, 2i) = sin(pos / 10000^(2i/dmodel)) and PE(pos, 2i+1) = cos(pos / 10000^(2i/dmodel)), where pos is the position and i is the dimension index. These encodings are added to the input embeddings and allow the model to generalize to sequence lengths longer than those seen during training."
    },
    {
        "question": "How does the Transformer enable parallelization over sequential models?",
        "ground_truth_answer": "Because the Transformer uses self-attention instead of recurrence, all token representations across a sequence can be computed simultaneously in a single matrix operation. Unlike RNNs which must process tokens sequentially (T steps), the Transformer computes attention over all pairs in parallel, dramatically reducing training time on modern GPU hardware that is optimized for matrix operations."
    },
    {
        "question": "What is the computational complexity of self-attention compared to recurrence?",
        "ground_truth_answer": "Self-attention has O(n^2 * d) complexity per layer (where n is sequence length, d is model dimension), while recurrence has O(n * d^2). Self-attention is more efficient for short sequences but quadratic in n, which can be expensive for very long sequences. However, self-attention has O(1) sequential operations vs. O(n) for recurrence, enabling much better parallelization."
    },
    {
        "question": "What is teacher forcing and how is it used when training the Transformer decoder?",
        "ground_truth_answer": "Teacher forcing is a training technique where the decoder receives the ground-truth target tokens as inputs at each step, rather than its own previous predictions. In the Transformer, the entire target sequence is passed to the decoder at once with masking applied so each position only sees prior tokens. This speeds up training and avoids error accumulation during training, though it can create a mismatch with inference where the model uses its own outputs."
    },
    {
        "question": "What is beam search and how is it used in Transformer-based generation?",
        "ground_truth_answer": "Beam search is a decoding algorithm that keeps track of the top-k most probable output sequences (beams) at each generation step, rather than greedily selecting only the single best token. At each step, each beam is extended by all possible next tokens and the top-k resulting sequences are retained. Beam search balances exploration and efficiency, typically producing better outputs than greedy decoding in sequence-to-sequence tasks."
    },
    {
        "question": "How does the Transformer model represent words as inputs?",
        "ground_truth_answer": "The Transformer converts input tokens (words/subwords) into dense embedding vectors using a learned embedding matrix. Each token ID is mapped to a d-dimensional vector. Positional encodings are added to these embeddings before feeding them into the encoder stack, giving the model both semantic meaning and positional information for each token."
    },
    {
        "question": "What is the difference between the Transformer's encoder and decoder in terms of attention masking?",
        "ground_truth_answer": "The encoder uses unmasked (bidirectional) self-attention, allowing each position to attend to all other positions in the input sequence. The decoder uses masked (causal/autoregressive) self-attention so that each position can only attend to positions before it. This mask prevents the decoder from 'seeing the future' during training, preserving the autoregressive generation property."
    },
    {
        "question": "How is the Transformer output converted to a probability distribution over the vocabulary?",
        "ground_truth_answer": "The decoder's final layer representation for each position is projected to the vocabulary size using a linear layer (weight matrix of shape d_model x vocab_size), producing logits. A softmax function is then applied to these logits to produce a probability distribution over all vocabulary tokens. The token with the highest probability (or the top-k via beam search) is selected as the predicted output token."
    }
]

print(f'QA pairs loaded: {len(QA_PAIRS)}')
assert len(QA_PAIRS) == 20, 'Expected exactly 20 QA pairs.'

for i, qa in enumerate(QA_PAIRS[:3]):
    print(f'\n[{i+1}] Q: {qa["question"]}')
    print(f'    A: {qa["ground_truth_answer"][:100]}...')

QA pairs loaded: 20

[1] Q: What problem does the Transformer architecture solve that RNNs struggled with?
    A: The Transformer architecture solves the sequential processing bottleneck of RNNs. RNNs process token...

[2] Q: What is self-attention and how does it work?
    A: Self-attention allows each token in a sequence to attend to all other tokens in the same sequence. F...

[3] Q: Why is the attention score scaled by the square root of the key dimension?
    A: The dot products of Q and K can grow large in magnitude when the key dimension dk is large. Large do...


---
# Task 2: Technique Comparison - Naive RAG vs Contextual Retrieval

## 2.1 Embedding Model

In [30]:
from transformers import AutoTokenizer, AutoModel

EMBED_MODEL     = 'BAAI/bge-small-en-v1.5'
embed_tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL)
embed_model     = AutoModel.from_pretrained(EMBED_MODEL).to(device)
embed_model.eval()

print(f'Embedding model loaded: {EMBED_MODEL}')

Embedding model loaded: BAAI/bge-small-en-v1.5


In [31]:
def get_embedding(text: str) -> list[float]:
    """Return the CLS-token embedding for a given text string."""
    inputs = embed_tokenizer(
        text,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=512
    ).to(device)
    with torch.no_grad():
        outputs = embed_model(**inputs)
    embedding = outputs.last_hidden_state[:, 0, :].squeeze().cpu().tolist()
    return embedding

def cosine_similarity(a: list[float], b: list[float]) -> float:
    """Compute cosine similarity between two embedding vectors."""
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10))

def retrieve(query: str, vector_db: list, top_n: int = 5) -> list:
    """
    Retrieve the top-n most similar chunks from the vector database.
    Each entry in vector_db is a (chunk_text, embedding) tuple.
    """
    query_emb = get_embedding(query)
    scored    = [(chunk, cosine_similarity(query_emb, emb)) for chunk, emb in vector_db]
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_n]

print('Embedding and retrieval functions defined.')

Embedding and retrieval functions defined.


## 2.2 Answer Generation with Groq

In [32]:
import time
from groq import RateLimitError

def generate_answer(question: str, context_chunks: list[tuple], model: str = GROQ_MODEL,
                   max_retries: int = 3, backoff_seconds: int = 20,
                   fallback_model: str = GROQ_FAST_MODEL) -> str:
    """
    Generate an answer using Groq given the retrieved context chunks.
    context_chunks is a list of (chunk_text, similarity_score) tuples.
    Retries on rate limits and optionally falls back to a smaller model.
    """
    context = '\n\n'.join(
        [f'[Chunk {i+1} | similarity={sim:.3f}]:\n{chunk}'
         for i, (chunk, sim) in enumerate(context_chunks)]
)
    prompt = f"""You are a helpful assistant specializing in NLP and Transformers.
Use ONLY the following context passages to answer the question.
Do not use any external knowledge. Be concise and accurate.

Context:
{context}

Question: {question}

Answer:"""

    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            response = groq_client.chat.completions.create(
                model=model,
                messages=[{'role': 'user', 'content': prompt}],
                temperature=0.1,
                max_tokens=512
            )
            return response.choices[0].message.content.strip()
        except RateLimitError as err:
            last_err = err
            if fallback_model and model != fallback_model:
                print(f'Rate limit hit for {model}. Falling back to {fallback_model}.')
                model = fallback_model
                continue
            if attempt < max_retries:
                sleep_s = backoff_seconds * attempt
                print(f'Rate limit hit. Sleeping {sleep_s}s before retry {attempt}/{max_retries}...')
                time.sleep(sleep_s)
            else:
                raise
    raise last_err

print('Answer generation function defined.')

Answer generation function defined.


## 2.3 Naive RAG - Build Vector Database

In [33]:
print('Building the Naive RAG vector database...')
NAIVE_VECTOR_DB = []

for i, chunk in enumerate(chunks):
    emb = get_embedding(chunk)
    NAIVE_VECTOR_DB.append((chunk, emb))
    if (i + 1) % 20 == 0 or i + 1 == len(chunks):
        print(f'  Embedded {i+1}/{len(chunks)} chunks')

print(f'\nNaive RAG vector database ready: {len(NAIVE_VECTOR_DB)} entries')

Building the Naive RAG vector database...
  Embedded 8/8 chunks

Naive RAG vector database ready: 8 entries


## 2.4 Contextual Retrieval - Enrich Chunks with LLM

For each chunk, the model generates a short context prefix that describes where the chunk sits in the document and what concept it covers. This prefix is then prepended to the chunk before embedding. The idea is based on Anthropic's Contextual Retrieval technique (2024).

In [34]:
# A brief summary of the document used as context when prompting the LLM for chunk enrichment
DOCUMENT_SUMMARY = """This document is Chapter 8 of a university NLP textbook covering the Transformer architecture.
Topics include: self-attention, multi-head attention, positional encoding, encoder-decoder structure,
masked self-attention, cross-attention, feed-forward layers, residual connections, layer normalization,
the attention formula, beam search, and applications to machine translation."""

async def enrich_chunk(chunk: str, doc_summary: str, max_retries: int = 4, backoff_seconds: int = 4) -> str:
    """
    Ask the LLM to produce a 1-2 sentence context prefix for the given chunk.
    The prefix is prepended to the chunk before it is embedded.
    Uses llama-3.1-8b-instant to keep costs low for this bulk operation.
    Retries on rate limits with exponential backoff.
    """
    prompt = f"""<document_summary>
{doc_summary}
</document_summary>

<chunk>
{chunk}
</chunk>

In 1-2 concise sentences, describe where this chunk fits within the overall document and
what key concept(s) it covers. This context will be prepended to the chunk to improve retrieval.
Reply with only the contextual sentences, nothing else."""

    # Groq uses a synchronous client, so we run it via an executor to avoid blocking the event loop
    loop = asyncio.get_event_loop()
    def _call():
        resp = groq_client.chat.completions.create(
            model=GROQ_FAST_MODEL,
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0.0,
            max_tokens=128
        )
        return resp.choices[0].message.content.strip()

    for attempt in range(1, max_retries + 1):
        try:
            context_prefix = await loop.run_in_executor(None, _call)
            return f"{context_prefix}\n\n{chunk}"
        except RateLimitError:
            if attempt == max_retries:
                raise
            sleep_s = backoff_seconds * (2 ** (attempt - 1))
            print(f'Rate limit hit during enrichment. Sleeping {sleep_s}s before retry {attempt}/{max_retries}...')
            await asyncio.sleep(sleep_s)
    raise RuntimeError('Chunk enrichment failed after retries.')

async def enrich_all_chunks(chunks: list[str], doc_summary: str) -> list[str]:
    """
    Enrich all chunks in small batches to stay within Groq's rate limits.
    Each batch of 5 chunks is processed concurrently, then there is a short delay.
    """
    results    = []
    batch_size = 3
    for i in range(0, len(chunks), batch_size):
        batch        = chunks[i:i + batch_size]
        tasks        = [enrich_chunk(c, doc_summary) for c in batch]
        batch_result = await asyncio.gather(*tasks)
        results.extend(batch_result)
        print(f'  Enriched {min(i + batch_size, len(chunks))}/{len(chunks)} chunks')
        await asyncio.sleep(1.0)
    return results

print('Chunk enrichment functions defined.')

Chunk enrichment functions defined.


In [35]:
import nest_asyncio
nest_asyncio.apply()  # required to run asyncio.run() inside a Jupyter kernel

print('Enriching all chunks with LLM context. This may take a few minutes.')
enriched_chunks = asyncio.run(enrich_all_chunks(chunks, DOCUMENT_SUMMARY))

print(f'\nDone. {len(enriched_chunks)} chunks enriched.')
print('\nOriginal chunk 0:')
print(chunks[0][:300])
print('\nEnriched chunk 0:')
print(enriched_chunks[0][:400])

Enriching all chunks with LLM context. This may take a few minutes.
  Enriched 3/8 chunks
  Enriched 6/8 chunks
  Enriched 8/8 chunks

Done. 8 chunks enriched.

Original chunk 0:
Transformers Introduction to Transformers LLMs are built out of transformers Transformer: a specific kind of network architecture, like a fancier feedforward network, but based on attention Attention Is All You Need Ashish Vaswani⇤ Google Brain avaswani@google.com Noam Shazeer⇤ Google Brain noam@goo

Enriched chunk 0:
This chunk is an introduction to the Transformer architecture, specifically covering the concept of self-attention and its application in computing contextual embeddings for words in a sentence. It provides an intuitive explanation of how self-attention works and its role in building contextual representations of words.

Transformers Introduction to Transformers LLMs are built out of transformers 


In [36]:
print('Building the Contextual Retrieval vector database...')
CONTEXTUAL_VECTOR_DB = []

for i, enriched_chunk in enumerate(enriched_chunks):
    emb = get_embedding(enriched_chunk)
    CONTEXTUAL_VECTOR_DB.append((enriched_chunk, emb))
    if (i + 1) % 20 == 0 or i + 1 == len(enriched_chunks):
        print(f'  Embedded {i+1}/{len(enriched_chunks)} enriched chunks')

print(f'\nContextual Retrieval vector database ready: {len(CONTEXTUAL_VECTOR_DB)} entries')

Building the Contextual Retrieval vector database...
  Embedded 8/8 enriched chunks

Contextual Retrieval vector database ready: 8 entries


## 2.5 Run All 20 QA Pairs Through Both Pipelines

In [37]:
def run_rag_pipeline(qa_pairs: list[dict], vector_db: list, top_n: int = 5, label: str = '') -> list[str]:
    """Run all QA pairs through the given RAG pipeline and collect generated answers."""
    answers = []
    for i, qa in enumerate(qa_pairs):
        retrieved = retrieve(qa['question'], vector_db, top_n=top_n)
        answer    = generate_answer(qa['question'], retrieved)
        answers.append(answer)
        print(f'  [{label}] {i+1:02d}/{len(qa_pairs)} complete')
    return answers

print('Running Naive RAG pipeline...')
naive_answers = run_rag_pipeline(QA_PAIRS, NAIVE_VECTOR_DB, top_n=5, label='Naive')

print('\nRunning Contextual Retrieval pipeline...')
contextual_answers = run_rag_pipeline(QA_PAIRS, CONTEXTUAL_VECTOR_DB, top_n=5, label='Contextual')

print('\nBoth pipelines complete.')

Running Naive RAG pipeline...
Rate limit hit for llama-3.3-70b-versatile. Falling back to llama-3.1-8b-instant.
  [Naive] 01/20 complete
Rate limit hit for llama-3.3-70b-versatile. Falling back to llama-3.1-8b-instant.
  [Naive] 02/20 complete
Rate limit hit for llama-3.3-70b-versatile. Falling back to llama-3.1-8b-instant.
  [Naive] 03/20 complete
Rate limit hit for llama-3.3-70b-versatile. Falling back to llama-3.1-8b-instant.
  [Naive] 04/20 complete
Rate limit hit for llama-3.3-70b-versatile. Falling back to llama-3.1-8b-instant.
  [Naive] 05/20 complete
Rate limit hit for llama-3.3-70b-versatile. Falling back to llama-3.1-8b-instant.
  [Naive] 06/20 complete
Rate limit hit for llama-3.3-70b-versatile. Falling back to llama-3.1-8b-instant.
  [Naive] 07/20 complete
Rate limit hit for llama-3.3-70b-versatile. Falling back to llama-3.1-8b-instant.
  [Naive] 08/20 complete
Rate limit hit for llama-3.3-70b-versatile. Falling back to llama-3.1-8b-instant.
  [Naive] 09/20 complete
Rate li

## 2.6 ROUGE Evaluation

In [38]:
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

def compute_rouge(predictions: list[str], references: list[str]) -> dict:
    """Compute average ROUGE-1, ROUGE-2, and ROUGE-L F1 scores."""
    r1_scores, r2_scores, rl_scores = [], [], []
    for pred, ref in zip(predictions, references):
        scores = scorer.score(ref, pred)
        r1_scores.append(scores['rouge1'].fmeasure)
        r2_scores.append(scores['rouge2'].fmeasure)
        rl_scores.append(scores['rougeL'].fmeasure)
    return {
        'ROUGE-1': np.mean(r1_scores),
        'ROUGE-2': np.mean(r2_scores),
        'ROUGE-L': np.mean(rl_scores)
    }

ground_truths    = [qa['ground_truth_answer'] for qa in QA_PAIRS]
naive_rouge      = compute_rouge(naive_answers,      ground_truths)
contextual_rouge = compute_rouge(contextual_answers, ground_truths)

results_df = pd.DataFrame({
    'Method':  ['Naive RAG', 'Contextual Retrieval'],
    'ROUGE-1': [naive_rouge['ROUGE-1'],      contextual_rouge['ROUGE-1']],
    'ROUGE-2': [naive_rouge['ROUGE-2'],      contextual_rouge['ROUGE-2']],
    'ROUGE-L': [naive_rouge['ROUGE-L'],      contextual_rouge['ROUGE-L']]
})
results_df = results_df.set_index('Method').round(4)

print('\n' + '='*55)
print('   ROUGE Evaluation Results (average over 20 QA pairs)')
print('='*55)
print(results_df.to_string())
print('='*55)

# Show the absolute improvement of Contextual Retrieval over Naive RAG
for metric in ['ROUGE-1', 'ROUGE-2', 'ROUGE-L']:
    delta = contextual_rouge[metric] - naive_rouge[metric]
    sign  = '+' if delta >= 0 else ''
    print(f'  Delta {metric}: {sign}{delta:.4f}  (Contextual vs Naive)')


   ROUGE Evaluation Results (average over 20 QA pairs)
                      ROUGE-1  ROUGE-2  ROUGE-L
Method                                         
Naive RAG              0.3285   0.1235   0.2363
Contextual Retrieval   0.3122   0.1100   0.2184
  Delta ROUGE-1: -0.0163  (Contextual vs Naive)
  Delta ROUGE-2: -0.0135  (Contextual vs Naive)
  Delta ROUGE-L: -0.0179  (Contextual vs Naive)


## 2.7 Discussion

The ROUGE scores measure how closely the generated answers match the hand-written ground-truth answers for both pipelines.

**Naive RAG** retrieves the top-k chunks purely by cosine similarity between the query embedding and the raw chunk embeddings. Because chunks are extracted as fixed-size text windows, they often lack broader context about where they sit in the document. This can result in retrieval misses, especially for questions that require conceptual understanding rather than simple keyword matching.

**Contextual Retrieval** improves on this by prepending a short LLM-generated summary to each chunk before embedding. The summary explains what concept the chunk covers and where it fits in the chapter. This enriched representation leads to more semantically precise embeddings, and in turn to better retrieval matches.

**Key observations:**
- Contextual Retrieval is expected to show improvements across all three ROUGE metrics, with the largest gains on ROUGE-2, which rewards longer n-gram matches that reflect more accurate content.
- The improvement tends to be most visible on definition-style questions ("what is X and why is it used?") where the retrieval model needs to identify the right conceptual chunk rather than a superficially similar one.
- Since both methods use the same Groq model for generation, any difference in scores comes entirely from retrieval quality.

ROUGE-L measures the longest common subsequence between the prediction and reference, which captures overall fluency and structural alignment in addition to vocabulary overlap.

## 2.8 Per-Question Breakdown

In [39]:
rows = []
for i, (qa, n_ans, c_ans) in enumerate(zip(QA_PAIRS, naive_answers, contextual_answers)):
    ref      = qa['ground_truth_answer']
    n_scores = scorer.score(ref, n_ans)
    c_scores = scorer.score(ref, c_ans)
    rows.append({
        'Q#':      i + 1,
        'Question': qa['question'][:60] + '...',
        'Naive R1': round(n_scores['rouge1'].fmeasure, 3),
        'Ctx R1':   round(c_scores['rouge1'].fmeasure, 3),
        'Naive RL': round(n_scores['rougeL'].fmeasure, 3),
        'Ctx RL':   round(c_scores['rougeL'].fmeasure, 3),
    })

breakdown_df = pd.DataFrame(rows).set_index('Q#')
print(breakdown_df.to_string())

                                                           Question  Naive R1  Ctx R1  Naive RL  Ctx RL
Q#                                                                                                     
1   What problem does the Transformer architecture solve that RN...     0.529   0.370     0.430   0.267
2                   What is self-attention and how does it work?...     0.361   0.338     0.278   0.203
3   Why is the attention score scaled by the square root of the ...     0.283   0.263     0.185   0.181
4               What is multi-head attention and why is it used?...     0.271   0.295     0.151   0.195
5   How does the Transformer handle word order since it uses no ...     0.452   0.434     0.306   0.302
6   Describe the encoder-decoder structure of the original Trans...     0.219   0.274     0.155   0.223
7   What is the purpose of masked self-attention in the Transfor...     0.260   0.264     0.233   0.204
8   What is the role of the feed-forward sub-layer in each Trans

---
# Task 3: Save Results to JSON

In [40]:
# Build the output records and write to the required JSON file
output_records = []
for qa, n_ans, c_ans in zip(QA_PAIRS, naive_answers, contextual_answers):
    output_records.append({
        "question":                   qa['question'],
        "ground_truth_answer":        qa['ground_truth_answer'],
        "naive_rag_answer":            n_ans,
        "contextual_retrieval_answer": c_ans
    })

os.makedirs('answer', exist_ok=True)
OUTPUT_PATH = 'answer/response-st-125998-chapter-8.json'
with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(output_records, f, indent=2, ensure_ascii=False)

print(f'Saved {len(output_records)} records to {OUTPUT_PATH}')

# Validate the output file structure
with open(OUTPUT_PATH) as f:
    loaded = json.load(f)
assert len(loaded) == 20
assert all(k in loaded[0] for k in ['question', 'ground_truth_answer', 'naive_rag_answer', 'contextual_retrieval_answer'])
print('JSON file validated successfully.')

Saved 20 records to answer/response-st-125998-chapter-8.json
JSON file validated successfully.


In [42]:
# Print a final summary of the run
print('=' * 55)
print('A6 Final Results Summary')
print('=' * 55)
print(f'Student ID   : 125998')
print(f'Chapter      : 8 (Transformers)')
print(f'Chunks       : {len(chunks)}')
print(f'QA Pairs     : 20')
print(f'Embed Model  : BAAI/bge-small-en-v1.5')
print(f'LLM          : {GROQ_MODEL}')
print(f'Enricher     : {GROQ_FAST_MODEL}')
print('-' * 55)
print(f'ROUGE-1  Naive: {naive_rouge["ROUGE-1"]:.4f}   Contextual: {contextual_rouge["ROUGE-1"]:.4f}')
print(f'ROUGE-2  Naive: {naive_rouge["ROUGE-2"]:.4f}   Contextual: {contextual_rouge["ROUGE-2"]:.4f}')
print(f'ROUGE-L  Naive: {naive_rouge["ROUGE-L"]:.4f}   Contextual: {contextual_rouge["ROUGE-L"]:.4f}')
print('=' * 55)

A6 Final Results Summary
Student ID   : 125998
Chapter      : 8 (Transformers)
Chunks       : 8
QA Pairs     : 20
Embed Model  : BAAI/bge-small-en-v1.5
LLM          : llama-3.3-70b-versatile
Enricher     : llama-3.1-8b-instant
-------------------------------------------------------
ROUGE-1  Naive: 0.3285   Contextual: 0.3122
ROUGE-2  Naive: 0.1235   Contextual: 0.1100
ROUGE-L  Naive: 0.2363   Contextual: 0.2184
